In [0]:
from pyspark.sql import SparkSession

passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]

columns = [
    "passenger_id",
    "passenger_name",
    "city",
    "travel_class",
    "country"
]

df_day1 = spark.createDataFrame(passengers_day1, columns)

display(df_day1)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(passengers_day2, columns)

display(df_day2)

passenger_id,passenger_name,city,travel_class,country
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
df_day1.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("passengers_delta")

In [0]:
%sql
SELECT * FROM passengers_delta;

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
spark.sql("""
SELECT COUNT(*) AS total_records
FROM passengers_delta
""").show()

+-------------+
|total_records|
+-------------+
|            5|
+-------------+



In [0]:
delta_df = spark.read.table("passengers_delta")
display(delta_df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
%sql DESCRIBE HISTORY passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T10:53:20.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1317101740172146),00cd9d43-2f4e-4f7b-9b69-fe0fd5c7f756,0617-103736-jag9ad1h-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1857)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "passengers_delta"
)

(
delta_table.alias("target")
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
)
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df_day2.createOrReplaceTempView("df_day2_view")

In [0]:
spark.sql("""
MERGE INTO passengers_delta t
USING df_day2_view s
ON t.passenger_id = s.passenger_id
WHEN MATCHED THEN UPDATE SET *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
MERGE INTO passengers_delta t
USING df_day2_view s
ON t.passenger_id = s.passenger_id
WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT passenger_id,
       passenger_name,
       travel_class
FROM passengers_delta
WHERE passenger_id = 102
""").show()

+------------+--------------+------------+
|passenger_id|passenger_name|travel_class|
+------------+--------------+------------+
|         102|   Priya Reddy| First Class|
+------------+--------------+------------+



In [0]:
version0 = spark.read \
    .format("delta") \
    .option("versionAsOf",0) \
    .table("passengers_delta")

display(version0)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:
latest = spark.read.table(
    "passengers_delta"
)
display(latest)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
103,Amit Kumar,Mumbai,Economy,India
105,Farhan Ali,Chennai,Economy,India
102,Priya Reddy,Bangalore,First Class,India
104,Sneha Patel,Hyderabad,Premium Economy,India
106,Neha Singh,Pune,Economy,India
107,Arjun Verma,Kochi,Business,India


In [0]:
version0.orderBy("passenger_id").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
latest.orderBy("passenger_id").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.table("passengers_delta") \
.filter("passenger_id=102") \
.show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.table("passengers_delta") \
.filter("passenger_id=104") \
.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
%sql
OPTIMIZE passengers_delta;

path,metrics
abfss://unity-catalog-storage@dbstoragez6ftkmhnls4ni.dfs.core.windows.net/7405610258530810/__unitystorage/catalogs/385bf242-fe73-4cd6-993f-ae57cd5f4e5b/tables/b446e32a-062b-4828-b673-4a34d422e58d,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781694481179, 1781694481689, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE passengers_delta
ZORDER BY (city);

path,metrics
abfss://unity-catalog-storage@dbstoragez6ftkmhnls4ni.dfs.core.windows.net/7405610258530810/__unitystorage/catalogs/385bf242-fe73-4cd6-993f-ae57cd5f4e5b/tables/b446e32a-062b-4828-b673-4a34d422e58d,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1908), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781694489959, 1781694490568, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
DELETE FROM passengers_delta
WHERE passenger_id = 105;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY passengers_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-06-17T11:08:24.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1317101740172146),062a9318-fb6e-42c7-a9c9-60d790a41c33,0617-103736-jag9ad1h-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1908, p25FileSize -> 1890, numDeletionVectorsRemoved -> 1, minFileSize -> 1890, numAddedFiles -> 1, maxFileSize -> 1890, p75FileSize -> 1890, p50FileSize -> 1890, numAddedBytes -> 1890)",null,Databricks-Runtime/18.2.x-photon-scala2.13
8,2026-06-17T11:08:23.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#14393L = 105)""])",null,List(1317101740172146),062a9318-fb6e-42c7-a9c9-60d790a41c33,0617-103736-jag9ad1h-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1789, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1414, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 372)",null,Databricks-Runtime/18.2.x-photon-scala2.13
7,2026-06-17T11:04:16.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#13674L = passenger_id#13175L)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1317101740172146),1f279855-99dd-4bd9-95f8-37e4bf905cf0,0617-103736-jag9ad1h-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 949, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 913)",null,Databricks-Runtime/18.2.x-photon-scala2.13
6,2026-06-17T11:04:07.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1317101740172146),9591967e-7a50-40b2-9695-aa40ed160312,0617-103736-jag9ad1h-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 8439, p25FileSize -> 1908, numDeletionVectorsRemoved -> 1, minFileSize -> 1908, numAddedFiles -> 1, maxFileSize -> 1908, p75FileSize -> 1908, p50FileSize -> 1908, numAddedBytes -> 1908)",null,Databricks-Runtime/18.2.x-photon-scala2.13
5,2026-06-17T11:04:05.000Z,142474429042272,azuser7224_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#13186L = passenger_id#13175L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1317101740172146),9591967e-7a50-40b2-9695-aa40ed160312,0617-103736-jag9ad1h-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 6531, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 3742, materializeSourceTimeMs -> 116, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1573, nu

In [0]:
%sql
VACUUM passengers_delta RETAIN 168 HOURS;

path
abfss://unity-catalog-storage@dbstoragez6ftkmhnls4ni.dfs.core.windows.net/7405610258530810/__unitystorage/catalogs/385bf242-fe73-4cd6-993f-ae57cd5f4e5b/tables/b446e32a-062b-4828-b673-4a34d422e58d
